In [2]:
import sys
import subprocess
import importlib

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--use-pep517",
    "--no-cache-dir",
    "--verbose",
    "seqeval==1.2.2",
]

result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        "seqeval installation failed. "
    )

importlib.invalidate_caches()

from seqeval.metrics import f1_score
from importlib.metadata import version

gold = [["B-PER", "I-PER", "O"]]
predicted = [["B-PER", "I-PER", "O"]]

assert f1_score(gold, predicted) == 1.0

print("\nSUCCESS: seqeval", version("seqeval"))
print("Import and metric check passed.")

Using pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 11.1 MB/s eta 0:00:00
  Installing build dependencies: started
  Running command pip subprocess to install build dependencies
  Using pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
  Non-user install by explicit request
  Created build tracker: /tmp/pip-build-tracker-nwzhfet9
  Entered build tracker: /tmp/pip-build-tracker-nwzhfet9
  Created temporary directory: /tmp/pip-install-_2nl4nvh
  Created temporary directory: /tmp/pip-ephem-wheel-cache-mb6tmfn9
  1 location(s) to search for versions of setuptools:
  * https://pypi.org/simple/setuptools/
  Fetching project page and analyzing links: https://pypi.org/simple/setuptools/
  Getting page https://pypi.org/simple/setuptools/
  Found index url https://pypi.org/simple/
  Looking up "https://pypi.org/simple/setuptools/" in the cache
  Request header has "max_age" as 0, cache 

In [3]:
import sys
import subprocess
import importlib
import importlib.util

dependencies = {
    "numpy": "numpy",
    "torch": "torch",
    "datasets": "datasets",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "seqeval": "seqeval",
    "sentencepiece": "sentencepiece",
}

missing_packages = [
    package
    for module, package in dependencies.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    print("Installing:", ", ".join(missing_packages))
    print("Upgrading pip and setuptools...")
    upgrade_pip_setuptools = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(upgrade_pip_setuptools.stdout)
    if upgrade_pip_setuptools.returncode != 0:
        raise RuntimeError(
            "Pip/Setuptools upgrade failed. "
        )

    installation = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            *missing_packages,
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    print(installation.stdout)

    if installation.returncode != 0:
        raise RuntimeError(
            "Dependency installation failed. "
        )

    importlib.invalidate_caches()
else:
    print("All required packages are already installed.")
# ------------------------------------------------------------
import zipfile
from pathlib import Path
from collections import Counter

import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoConfig
SEED = 42
MAX_INPUT_LENGTH = 512

MODEL_CHECKPOINTS = {
    "xlm-r": "xlm-roberta-base",
    "parsbert": "HooshvareLab/bert-fa-base-uncased",
    "tookabert": "PartAI/TookaBERT-Base",
}

ENTITY_TYPES = ["PER", "ORG", "LOC", "FAC", "EVENT", "PROD"]

label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-FAC", "I-FAC",
    "B-EVENT", "I-EVENT",
    "B-PROD", "I-PROD",
]

label2id = {
    label: index
    for index, label in enumerate(label_list)
}

id2label = {
    index: label
    for label, index in label2id.items()
}

raw_to_standard = {
    "O": "O",
    "B-PERS": "B-PER",
    "I-PERS": "I-PER",
    "B-ORG": "B-ORG",
    "I-ORG": "I-ORG",
    "B-LOC": "B-LOC",
    "I-LOC": "I-LOC",
    "B-FAC": "B-FAC",
    "I-FAC": "I-FAC",
    "B-EVENT": "B-EVENT",
    "I-EVENT": "I-EVENT",
    "B-PRO": "B-PROD",
    "I-PRO": "I-PROD",
}

print("\nPyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------

repository_dir = Path("PersianNER")
extraction_dir = Path("ArmanPersoNERCorpus_extracted")

if not repository_dir.exists():
    print("\nDownloading dataset repository...")

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/HaniehP/PersianNER.git",
            str(repository_dir),
        ],
        check=True,
    )

def locate_dataset_file(filename):
    matches = sorted(extraction_dir.rglob(filename))

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple copies of {filename} found: {matches}"
        )

    return matches[0] if matches else None

required_files = ["train_fold1.txt", "test_fold1.txt"]

if any(
    locate_dataset_file(filename) is None
    for filename in required_files
):
    archive_path = repository_dir / "ArmanPersoNERCorpus.zip"

    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Missing archive: {archive_path.resolve()}"
        )

    print("Extracting dataset...")
    extraction_dir.mkdir(parents=True, exist_ok=True)
    destination = extraction_dir.resolve()

    with zipfile.ZipFile(archive_path, "r") as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()

            if not target.is_relative_to(destination):
                raise ValueError(
                    f"Unsafe archive entry: {member.filename}"
                )

        archive.extractall(extraction_dir)

dataset_paths = {
    filename: locate_dataset_file(filename)
    for filename in required_files
}

if any(path is None for path in dataset_paths.values()):
    raise FileNotFoundError(
        "Both Fold 1 files must be present after extraction."
    )

# ------------------------------------------------------------
def parse_arman_file(filepath):
    sentences = []
    tokens = []
    tags = []

    with Path(filepath).open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                if tokens:
                    sentences.append({
                        "tokens": tokens,
                        "ner_tags": tags,
                    })
                    tokens = []
                    tags = []
                continue

            parts = line.split()

            if len(parts) != 2:
                raise ValueError(
                    f"{filepath}, line {line_number}: "
                    f"expected two fields, found {len(parts)}."
                )

            token, raw_tag = parts
            raw_tag = raw_tag.upper()

            if raw_tag not in raw_to_standard:
                raise ValueError(
                    f"{filepath}, line {line_number}: "
                    f"unknown label {raw_tag!r}."
                )

            standard_tag = raw_to_standard[raw_tag]
            tokens.append(token)
            tags.append(label2id[standard_tag])

    if tokens:
        sentences.append({
            "tokens": tokens,
            "ner_tags": tags,
        })

    for sentence_number, sentence in enumerate(sentences, start=1):
        assert len(sentence["tokens"]) == len(sentence["ner_tags"])

        previous_tag = "O"

        for tag_id in sentence["ner_tags"]:
            tag = id2label[tag_id]

            if tag.startswith("I-"):
                entity_type = tag[2:]

                if previous_tag not in {
                    f"B-{entity_type}",
                    f"I-{entity_type}",
                }:
                    raise ValueError(
                        f"{filepath}, sentence {sentence_number}: "
                        f"invalid BIO transition "
                        f"{previous_tag} -> {tag}."
                    )

            previous_tag = tag

    return sentences

def count_labels(sentences):
    return Counter(
        id2label[int(tag)]
        for sentence in sentences
        for tag in sentence["ner_tags"]
    )

raw_train = parse_arman_file(dataset_paths["train_fold1.txt"])
raw_test = parse_arman_file(dataset_paths["test_fold1.txt"])

train_counts = count_labels(raw_train)
test_counts = count_labels(raw_test)

assert len(raw_train) == 5121
assert len(raw_test) == 2560

assert sum(train_counts.values()) == 167877
assert sum(test_counts.values()) == 82119

assert train_counts["B-PROD"] == 459
assert train_counts["I-PROD"] == 408
assert test_counts["B-PROD"] == 265
assert test_counts["I-PROD"] == 331

assert train_counts["O"] == 151182
assert test_counts["O"] == 73787

print("\nRaw labels verified.")

# ------------------------------------------------------------
def add_sentence_ids(sentences, prefix):
    return [
        {
            "sentence_id": f"{prefix}_{index:05d}",
            "tokens": sentence["tokens"],
            "ner_tags": sentence["ner_tags"],
        }
        for index, sentence in enumerate(sentences)
    ]

full_training_data = Dataset.from_list(
    add_sentence_ids(raw_train, "trainfile")
)

test_data = Dataset.from_list(
    add_sentence_ids(raw_test, "testfile")
)

train_validation = full_training_data.train_test_split(
    test_size=0.10,
    seed=SEED,
    shuffle=True,
)

split_dataset = DatasetDict({
    "train": train_validation["train"],
    "validation": train_validation["test"],
    "test": test_data,
})

# -----------------------------------------------------------

def sentence_key(row):
    return tuple(row["tokens"])

exclusion_log = []

def log_exclusion(split_name, row, reason):
    exclusion_log.append({
        "split": split_name,
        "sentence_id": row["sentence_id"],
        "reason": reason,
        "word_count": len(row["tokens"]),
        "text": " ".join(row["tokens"]),
    })

test_keys = {
    sentence_key(row)
    for row in split_dataset["test"]
}

validation_keep = []

for index, row in enumerate(split_dataset["validation"]):
    if sentence_key(row) in test_keys:
        log_exclusion(
            "validation",
            row,
            "Exact token sequence also occurs in test",
        )
    else:
        validation_keep.append(index)

clean_validation = split_dataset["validation"].select(
    validation_keep
)

validation_keys = {
    sentence_key(row)
    for row in clean_validation
}

training_keep = []
seen_training = {}

for index, row in enumerate(split_dataset["train"]):
    key = sentence_key(row)
    labels = tuple(row["ner_tags"])

    if key in test_keys:
        log_exclusion(
            "train",
            row,
            "Exact token sequence also occurs in test",
        )
        continue

    if key in validation_keys:
        log_exclusion(
            "train",
            row,
            "Exact token sequence also occurs in validation",
        )
        continue

    if key in seen_training:
        previous = seen_training[key]

        if labels != previous["labels"]:
            raise ValueError(
                "Conflicting training duplicates: "
                f"{previous['sentence_id']} and "
                f"{row['sentence_id']}."
            )

        log_exclusion(
            "train",
            row,
            "Identical training duplicate; kept "
            + previous["sentence_id"],
        )
        continue

    seen_training[key] = {
        "labels": labels,
        "sentence_id": row["sentence_id"],
    }
    training_keep.append(index)

clean_split_dataset = DatasetDict({
    "train": split_dataset["train"].select(training_keep),
    "validation": clean_validation,
    "test": split_dataset["test"],
})

# ------------------------------------------------------------
clean_keys = {
    name: {sentence_key(row) for row in dataset}
    for name, dataset in clean_split_dataset.items()
}

for left, right in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    assert not (clean_keys[left] & clean_keys[right]), (
        f"Overlap remains between {left} and {right}."
    )

assert len(clean_keys["train"]) == len(clean_split_dataset["train"])

assert len(clean_split_dataset["test"]) == len(split_dataset["test"])

for original, retained in zip(
    split_dataset["test"],
    clean_split_dataset["test"],
):
    assert original == retained

expected_sizes = {
    "train": (4598, 151315),
    "validation": (512, 16330),
    "test": (2560, 82119),
}

print("\n" + "=" * 70)
print("CLEAN SPLIT SUMMARY")
print("=" * 70)

for split_name, dataset in clean_split_dataset.items():
    word_count = sum(len(row["tokens"]) for row in dataset)
    counts = count_labels(dataset)

    expected_sentences, expected_words = expected_sizes[split_name]

    assert len(dataset) == expected_sentences
    assert word_count == expected_words

    for entity_type in ENTITY_TYPES:
        assert counts[f"B-{entity_type}"] > 0

    print(
        f"{split_name.upper()}: "
        f"{len(dataset):,} sentences | {word_count:,} words"
    )

assert len(exclusion_log) == 11
print(f"Excluded development rows: {len(exclusion_log)}")

# ------------------------------------------------------------
audit_tokenizers = {}
audit_configs = {}

for model_name, checkpoint in MODEL_CHECKPOINTS.items():
    print(f"\nLoading tokenizer/configuration: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint,
        use_fast=True,
    )
    config = AutoConfig.from_pretrained(checkpoint)

    if not tokenizer.is_fast:
        raise RuntimeError(
            f"{model_name}: fast word-ID alignment is required."
        )

    if tokenizer.unk_token_id is None:
        raise RuntimeError(
            f"{model_name}: an existing unknown token is required."
        )

    if config.model_type == "xlm-roberta":
        usable_length = (
            config.max_position_embeddings
            - config.pad_token_id
            - 1
        )
    elif config.model_type == "bert":
        usable_length = config.max_position_embeddings
    else:
        raise ValueError(
            f"Unexpected architecture: {config.model_type}"
        )

    if usable_length < MAX_INPUT_LENGTH:
        raise ValueError(
            f"{model_name} supports only {usable_length} positions."
        )

    tokenizer.model_max_length = MAX_INPUT_LENGTH
    tokenizer.padding_side = "right"

    audit_tokenizers[model_name] = tokenizer
    audit_configs[model_name] = config

# ------------------------------------------------------------
def make_encoder(tokenizer, model_name):

    def encode_sentence(row):
        words = row["tokens"]
        gold_labels = row["ner_tags"]

        encoded = tokenizer(
            words,
            is_split_into_words=True,
            truncation=False,
            padding=False,
            add_special_tokens=True,
            return_attention_mask=True,
        )

        original_word_ids = encoded.word_ids()

        fields = ["input_ids", "attention_mask"]
        if "token_type_ids" in encoded:
            fields.append("token_type_ids")

        output = {field: [] for field in fields}
        aligned_word_ids = []
        fallback_mask = []

        content_positions = [
            position
            for position, word_id in enumerate(original_word_ids)
            if word_id is not None
        ]

        if not content_positions:
            raise ValueError(
                f"{model_name}: {row['sentence_id']} has no "
                "represented words. Inspect before continuing."
            )

        trailing_special_start = content_positions[-1] + 1
        next_unseen_word = 0

        def append_fallback(word_id):
            output["input_ids"].append(tokenizer.unk_token_id)
            output["attention_mask"].append(1)

            if "token_type_ids" in output:
                output["token_type_ids"].append(0)

            aligned_word_ids.append(word_id)
            fallback_mask.append(True)

        for position, word_id in enumerate(original_word_ids):

            if position == trailing_special_start:
                while next_unseen_word < len(words):
                    append_fallback(next_unseen_word)
                    next_unseen_word += 1

            if word_id is not None:
                while next_unseen_word < word_id:
                    append_fallback(next_unseen_word)
                    next_unseen_word += 1

                next_unseen_word = max(
                    next_unseen_word,
                    word_id + 1,
                )

            for field in fields:
                output[field].append(encoded[field][position])

            aligned_word_ids.append(
                -1 if word_id is None else word_id
            )
            fallback_mask.append(False)

        while next_unseen_word < len(words):
            append_fallback(next_unseen_word)
            next_unseen_word += 1

        labels = []
        seen_words = set()

        for word_id in aligned_word_ids:
            if word_id == -1 or word_id in seen_words:
                labels.append(-100)
            else:
                labels.append(int(gold_labels[word_id]))
                seen_words.add(word_id)

        output["labels"] = labels

        assert seen_words == set(range(len(words)))

        represented_order = [
            word_id
            for word_id in aligned_word_ids
            if word_id != -1
        ]
        assert represented_order == sorted(represented_order)

        assert [
            label for label in labels if label != -100
        ] == list(gold_labels)

        sequence_length = len(output["input_ids"])

        assert all(
            len(values) == sequence_length
            for values in output.values()
        )

        if sequence_length > MAX_INPUT_LENGTH:
            raise ValueError(
                f"{model_name}: {row['sentence_id']} has "
                f"{sequence_length} tokens. Stopped without truncation."
            )

        output["sentence_id"] = row["sentence_id"]
        output["word_ids"] = aligned_word_ids
        output["fallback_mask"] = fallback_mask

        return output

    return encode_sentence

# ------------------------------------------------------------
revised_tokenized_datasets = {}
revised_alignment = {}
revised_tokenization_summary = {}

expected_tooka_fallbacks = {
    "train": 554,
    "validation": 54,
    "test": 320,
}

for model_name, tokenizer in audit_tokenizers.items():
    encoder = make_encoder(tokenizer, model_name)

    model_splits = {}
    alignment_splits = {}

    print("\n" + "=" * 70)
    print(f"TOKENIZING: {model_name.upper()}")
    print("=" * 70)

    for split_name, dataset in clean_split_dataset.items():
        encoded_dataset = dataset.map(
            encoder,
            remove_columns=dataset.column_names,
            load_from_cache_file=False,
            desc=f"{model_name}: {split_name}",
        )

        metadata_columns = [
            "sentence_id",
            "word_ids",
            "fallback_mask",
        ]

        alignment_splits[split_name] = (
            encoded_dataset.remove_columns([
                column
                for column in encoded_dataset.column_names
                if column not in metadata_columns
            ])
        )

        model_splits[split_name] = encoded_dataset.remove_columns(
            metadata_columns
        )

        original_words = sum(
            len(row["tokens"]) for row in dataset
        )

        supervised_words = sum(
            sum(label != -100 for label in labels)
            for labels in encoded_dataset["labels"]
        )

        fallback_count = sum(
            sum(mask)
            for mask in encoded_dataset["fallback_mask"]
        )

        maximum_length = max(
            len(ids)
            for ids in encoded_dataset["input_ids"]
        )

        original_counts = Counter(
            int(label)
            for row in dataset
            for label in row["ner_tags"]
        )

        encoded_counts = Counter(
            int(label)
            for labels in encoded_dataset["labels"]
            for label in labels
            if label != -100
        )

        assert supervised_words == original_words
        assert encoded_counts == original_counts

        expected_fallbacks = (
            expected_tooka_fallbacks[split_name]
            if model_name == "tookabert"
            else 0
        )

        assert fallback_count == expected_fallbacks, (
            f"{model_name}/{split_name}: fallback count changed. "
            "Inspect before training."
        )

        revised_tokenization_summary[
            (model_name, split_name)
        ] = {
            "sentences": len(encoded_dataset),
            "original_words": original_words,
            "supervised_words": supervised_words,
            "fallback_tokens": fallback_count,
            "maximum_sequence_length": maximum_length,
        }

        print(
            f"{split_name.upper()}: "
            f"{supervised_words:,}/{original_words:,} words | "
            f"{fallback_count} fallbacks | "
            f"max length {maximum_length}"
        )

    revised_tokenized_datasets[model_name] = DatasetDict(
        model_splits
    )

    revised_alignment[model_name] = DatasetDict(
        alignment_splits
    )

# ------------------------------------------------------------
print("\n" + "=" * 70)
print("REVISED DATA READY")
print("=" * 70)
print("Product labels correctly mapped.")
print("All six classes present in all splits.")
print("Exact cross-split overlaps removed.")
print("Duplicate training sentences removed.")
print("Official test set unchanged.")
print("All original words and gold labels preserved.")
print("No truncation.")
print("Fallback and word-alignment metadata retained.")
print("No model weights loaded.")
print("No models trained.")


All required packages are already installed.

PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4

Extracting dataset...

Raw labels verified.

CLEAN SPLIT SUMMARY
TRAIN: 4,598 sentences | 151,315 words
VALIDATION: 512 sentences | 16,330 words
TEST: 2,560 sentences | 82,119 words
Excluded development rows: 11

Loading tokenizer/configuration: xlm-r


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

TRAIN: 151,315/151,315 words | 554 fallbacks | max length 256


tookabert: validation:   0%|          | 0/512 [00:00<?, ? examples/s]

VALIDATION: 16,330/16,330 words | 54 fallbacks | max length 220


tookabert: test:   0%|          | 0/2560 [00:00<?, ? examples/s]

TEST: 82,119/82,119 words | 320 fallbacks | max length 302

REVISED DATA READY
Product labels correctly mapped.
All six classes present in all splits.
Exact cross-split overlaps removed.
Duplicate training sentences removed.
Official test set unchanged.
All original words and gold labels preserved.
No truncation.
Fallback and word-alignment metadata retained.
No model weights loaded.
No models trained.


In [4]:
import gc
import json
import csv
import sys
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from collections import Counter

import numpy as np
import torch
from transformers import (
    AutoModelForTokenClassification, TrainingArguments, Trainer,
    DataCollatorForTokenClassification, set_seed,
)


SEED = 42
LEARNING_RATE =2e-5
EPOCHS = 4
TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
EVAL_BATCH_SIZE = 4
SAVE_TO_GOOGLE_DRIVE = True
MODEL_ORDER = ["xlm-r", "parsbert", "tookabert"]
CLASSES = ["PER", "ORG", "LOC", "FAC", "EVENT", "PROD"]
BIO_POLICY = (
    "Gold must be valid IOB2. In predictions, an I-TYPE without an active "
    "entity of the same type starts a new entity. Raw predictions are retained. "
    "Spans use zero-based start and exclusive end. Undefined P/R/F1 = 0. "
    "Macro scores average all six named classes, excluding O."
)

required = [
    "clean_split_dataset", "revised_tokenized_datasets", "revised_alignment",
    "audit_tokenizers", "audit_configs", "MODEL_CHECKPOINTS", "label_list",
    "label2id", "id2label", "exclusion_log",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError("Run successfully. Missing: " + ", ".join(missing))
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("This cell is configured for one GPU, as in a standard Colab runtime.")
assert label2id == {label: i for i, label in enumerate(label_list)}
assert {int(i): label for i, label in id2label.items()} == dict(enumerate(label_list))
assert set(label_list) == {"O"} | {f"{p}-{c}" for c in CLASSES for p in ("B", "I")}

def extract_spans(tags, strict_gold=False):
    spans = set()
    start, active_type = None, None
    invalid = 0
    for index, tag in enumerate(list(tags) + ["O"]):
        if tag == "O":
            prefix, entity_type = "O", None
        else:
            prefix, entity_type = tag.split("-", 1)
            if prefix not in ("B", "I") or entity_type not in CLASSES:
                raise ValueError(f"Unrecognized BIO label: {tag}")
        continues = prefix == "I" and active_type == entity_type
        if prefix == "I" and not continues:
            invalid += 1
            if strict_gold:
                raise ValueError(f"Invalid gold BIO transition at word {index}: {tag}")
        if active_type is not None and not continues:
            spans.add((start, index, active_type))
            start, active_type = None, None
        if prefix != "O" and not continues:
            start, active_type = index, entity_type
    return spans, invalid


def prf(tp, predicted, gold):
    precision = tp / predicted if predicted else 0.0
    recall = tp / gold if gold else 0.0
    f1 = 2 * tp / (predicted + gold) if predicted + gold else 0.0
    return {"precision": precision, "recall": recall, "f1": f1,
            "support": gold, "predicted": predicted, "true_positives": tp}


def score_sequences(gold_sequences, prediction_sequences):
    if len(gold_sequences) != len(prediction_sequences):
        raise ValueError("Different numbers of gold and predicted sentences.")
    gold_spans, predicted_spans = set(), set()
    correct_words = total_words = invalid = 0
    for sid, (gold, predicted) in enumerate(zip(gold_sequences, prediction_sequences)):
        if len(gold) != len(predicted):
            raise ValueError(f"Missing predictions in sentence {sid}.")
        gs, _ = extract_spans(gold, strict_gold=True)
        ps, n_invalid = extract_spans(predicted)
        gold_spans.update((sid, *span) for span in gs)
        predicted_spans.update((sid, *span) for span in ps)
        invalid += n_invalid
        correct_words += sum(g == p for g, p in zip(gold, predicted))
        total_words += len(gold)
    typed = prf(len(gold_spans & predicted_spans), len(predicted_spans), len(gold_spans))
    gold_boundaries = {span[:3] for span in gold_spans}
    predicted_boundaries = {span[:3] for span in predicted_spans}
    boundary = prf(len(gold_boundaries & predicted_boundaries),
                   len(predicted_boundaries), len(gold_boundaries))
    per_class = {}
    for entity_type in CLASSES:
        gs = {span for span in gold_spans if span[-1] == entity_type}
        ps = {span for span in predicted_spans if span[-1] == entity_type}
        per_class[entity_type] = prf(len(gs & ps), len(ps), len(gs))
    macro = {metric: sum(row[metric] for row in per_class.values()) / len(CLASSES)
             for metric in ("precision", "recall", "f1")}
    return {"typed_micro": typed, "boundary_micro": boundary, "typed_macro": macro,
            "per_class": per_class, "token_accuracy": correct_words / total_words if total_words else 0.0,
            "word_count": total_words, "invalid_predicted_bio_transitions": invalid}


def decode_output(predictions, labels):
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.asarray(predictions)
    if predictions.ndim == 3:
        predictions = predictions.argmax(axis=-1)
    if predictions.shape != labels.shape:
        raise ValueError("Prediction/label array shapes differ.")
    gold, predicted = [], []
    for pred_row, gold_row in zip(predictions, labels):
        keep = gold_row != -100
        gold.append([label_list[int(i)] for i in gold_row[keep]])
        predicted.append([label_list[int(i)] for i in pred_row[keep]])
    return gold, predicted


def compute_metrics(output):
    gold, predicted = decode_output(output.predictions, output.label_ids)
    scores = score_sequences(gold, predicted)
    return {
        "typed_micro_precision": scores["typed_micro"]["precision"],
        "typed_micro_recall": scores["typed_micro"]["recall"],
        "typed_micro_f1": scores["typed_micro"]["f1"],
        "typed_macro_f1": scores["typed_macro"]["f1"],
        "boundary_micro_f1": scores["boundary_micro"]["f1"],
        "token_accuracy": scores["token_accuracy"],
    }


def retain_prediction_ids(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

for model_key in MODEL_ORDER:
    if audit_tokenizers[model_key].padding_side != "right":
        raise ValueError("Right padding is required for this alignment check.")
    for split in ("train", "validation", "test"):
        raw = clean_split_dataset[split]
        encoded = revised_tokenized_datasets[model_key][split]
        alignment = revised_alignment[model_key][split]
        assert len(raw) == len(encoded) == len(alignment)
        for row, inputs, metadata in zip(raw, encoded, alignment):
            assert row["sentence_id"] == metadata["sentence_id"]
            assert len(inputs["input_ids"]) == len(inputs["labels"]) == len(metadata["word_ids"]) == len(metadata["fallback_mask"])
            positions = [i for i, label in enumerate(inputs["labels"]) if label != -100]
            assert [metadata["word_ids"][i] for i in positions] == list(range(len(row["tokens"])))
            assert [inputs["labels"][i] for i in positions] == list(row["ner_tags"])
            assert len(inputs["input_ids"]) <= 512
            extract_spans([label_list[int(i)] for i in row["ner_tags"]], strict_gold=True)
print("All models: original words, labels, sentence order, and alignment verified.")

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    storage_root = Path("/content/drive/MyDrive/Persian_NER_Revised")
else:
    storage_root = Path.cwd() / "Persian_NER_Revised"
run_id = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
REVISED_RUN_DIR = storage_root / run_id
REVISED_RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Saving this run to:", REVISED_RUN_DIR)


def save_json(path, value):
    def convert(obj):
        if isinstance(obj, np.generic):
            return obj.item()
        if isinstance(obj, Path):
            return str(obj)
        raise TypeError(f"Not JSON serializable: {type(obj)}")
    with Path(path).open("w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=convert)

versions = {"python": sys.version, "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0)}
for package in ("torch", "transformers", "datasets", "accelerate", "numpy", "tokenizers", "sentencepiece"):
    try:
        versions[package] = version(package)
    except PackageNotFoundError:
        versions[package] = "not installed"
save_json(REVISED_RUN_DIR / "environment.json", versions)
save_json(REVISED_RUN_DIR / "label_map.json", {"labels": label_list, "label2id": label2id})
save_json(REVISED_RUN_DIR / "exclusions.json", exclusion_log)
save_json(REVISED_RUN_DIR / "evaluation_policy.json", {"bio_policy": BIO_POLICY, "macro_classes": CLASSES})
clean_split_dataset.save_to_disk(str(REVISED_RUN_DIR / "clean_splits"))

def export_split(trainer, model_key, split, model_dir):
    dataset = revised_tokenized_datasets[model_key][split]
    output = trainer.predict(dataset, metric_key_prefix=split)
    gold, predicted = decode_output(output.predictions, output.label_ids)
    raw = clean_split_dataset[split]
    alignment = revised_alignment[model_key][split]
    assert len(gold) == len(predicted) == len(raw)
    export_path = model_dir / f"{split}_word_predictions.jsonl"
    with export_path.open("w", encoding="utf-8") as handle:
        for row, metadata, gs, ps in zip(raw, alignment, gold, predicted):
            assert row["sentence_id"] == metadata["sentence_id"]
            assert gs == [label_list[int(i)] for i in row["ner_tags"]]
            assert len(gs) == len(ps) == len(row["tokens"])
            counts = Counter(int(i) for i in metadata["word_ids"] if i >= 0)
            fallback_words = {int(w) for w, f in zip(metadata["word_ids"], metadata["fallback_mask"]) if f and w >= 0}
            record = {
                "sentence_id": row["sentence_id"], "tokens": row["tokens"],
                "gold_labels": gs, "predicted_labels": ps,
                "input_subwords_per_word": [counts[i] for i in range(len(gs))],
                "fallback_word_mask": [i in fallback_words for i in range(len(gs))],
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    scores = score_sequences(gold, predicted)
    save_json(model_dir / f"{split}_scores.json", scores)
    save_json(model_dir / f"{split}_trainer_metrics.json", output.metrics)
    with (model_dir / f"{split}_per_class.csv").open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["class", "precision", "recall", "f1", "support", "predicted", "true_positives"])
        writer.writeheader()
        writer.writerows({"class": c, **scores["per_class"][c]} for c in CLASSES)
    np.savez_compressed(model_dir / f"{split}_predictions.npz",
                        predictions=output.predictions, labels=output.label_ids)
    print(f"{model_key} {split}: typed micro F1={scores['typed_micro']['f1']:.4f}, "
          f"macro F1={scores['typed_macro']['f1']:.4f}, "
          f"boundary F1={scores['boundary_micro']['f1']:.4f}")
    return scores, str(export_path)

test_predictions_store = {}


def train_and_export_model(model_key):
    model_dir = REVISED_RUN_DIR / model_key
    model_dir.mkdir(exist_ok=False)
    model = trainer = None
    try:
        tokenizer = audit_tokenizers[model_key]
        checkpoint = MODEL_CHECKPOINTS[model_key]
        revision = getattr(audit_configs[model_key], "_commit_hash", None)
        revision_args = {"revision": revision} if revision else {}
        tokenizer.save_pretrained(str(model_dir / "tokenizer"))
        revised_alignment[model_key].save_to_disk(str(model_dir / "alignment"))
        revised_tokenized_datasets[model_key].save_to_disk(str(model_dir / "tokenized_inputs"))
        tokenizer_hash = hashlib.sha256(tokenizer.backend_tokenizer.to_str().encode("utf-8")).hexdigest()
        save_json(model_dir / "source.json", {
            "checkpoint": checkpoint, "model_revision": revision,
            "tokenizer_revision": tokenizer.init_kwargs.get("_commit_hash"),
            "tokenizer_sha256": tokenizer_hash, "seed": SEED,
            "selection_metric": "validation typed_micro_f1", "bio_policy": BIO_POLICY,
        })
        set_seed(SEED)
        model = AutoModelForTokenClassification.from_pretrained(
            checkpoint, num_labels=len(label_list), id2label=id2label,
            label2id=label2id, **revision_args,
        )
        args = TrainingArguments(
            output_dir=str(model_dir / "checkpoints"),
            eval_strategy="epoch", save_strategy="epoch",
            learning_rate=LEARNING_RATE, num_train_epochs=EPOCHS,
            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            weight_decay=0.01, optim="adamw_torch",
            lr_scheduler_type="linear", warmup_steps=0,
            load_best_model_at_end=True, metric_for_best_model="typed_micro_f1",
            greater_is_better=True, save_total_limit=2,
            seed=SEED, data_seed=SEED,
            fp16=False, bf16=False,
            logging_steps=50, report_to="none", push_to_hub=False,
            eval_accumulation_steps=8, dataloader_num_workers=0,
        )
        save_json(model_dir / "training_arguments.json", args.to_dict())
        trainer = Trainer(
            model=model, args=args,
            train_dataset=revised_tokenized_datasets[model_key]["train"],
            eval_dataset=revised_tokenized_datasets[model_key]["validation"],
            processing_class=tokenizer,
            data_collator=DataCollatorForTokenClassification(tokenizer, padding=True, label_pad_token_id=-100),
            compute_metrics=compute_metrics,
            preprocess_logits_for_metrics=retain_prediction_ids,
        )
        print(f"\nTraining {model_key} from {checkpoint}")
        trained = trainer.train()
        best_dir = model_dir / "best_model"
        trainer.save_model(str(best_dir))
        tokenizer.save_pretrained(str(best_dir))
        trainer.save_state()
        save_json(model_dir / "training_metrics.json", trained.metrics)
        save_json(model_dir / "training_history.json", trainer.state.log_history)
        save_json(model_dir / "selected_checkpoint.json", {
            "best_checkpoint": trainer.state.best_model_checkpoint,
            "best_validation_typed_micro_f1": trainer.state.best_metric,
        })
        validation_scores, _ = export_split(trainer, model_key, "validation", model_dir)
        test_scores, test_file = export_split(trainer, model_key, "test", model_dir)
        result = {"best_model_dir": str(best_dir), "predictions_file": test_file,
                  "validation_scores": validation_scores, "test_scores": test_scores}
        save_json(model_dir / "completed.json", result)
        return result
    finally:
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()

def run_one_model(model_key):
    completed_path = REVISED_RUN_DIR / model_key / "completed.json"
    if completed_path.is_file():
        with completed_path.open(encoding="utf-8") as handle:
            result = json.load(handle)
        print(f"{model_key}: already complete; reusing saved results.")
    else:
        if (REVISED_RUN_DIR / model_key).exists():
            raise RuntimeError(
                f"An incomplete {model_key} run exists at {REVISED_RUN_DIR / model_key}. "
                "Its checkpoints are preserved. Automatic resume is not implemented."
            )
        result = train_and_export_model(model_key)
    test_predictions_store[model_key] = result
    save_json(REVISED_RUN_DIR / "completed_models.json", test_predictions_store)
    print("Best model:", result["best_model_dir"])
    return result

print("Setup complete. No model trained by this setup cell.")


All models: original words, labels, sentence order, and alignment verified.
Mounted at /content/drive
Saving this run to: /content/drive/MyDrive/Persian_NER_Revised/run_20260929T190017_726808Z


Saving the dataset (0/1 shards):   0%|          | 0/4598 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/512 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2560 [00:00<?, ? examples/s]

Setup complete. No model trained by this setup cell.


In [5]:
result_xlm_r = run_one_model("xlm-r")

Saving the dataset (0/1 shards):   0%|          | 0/4598 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/512 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2560 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4598 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/512 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

xlm-r validation: typed micro F1=0.8048, macro F1=0.7333, boundary F1=0.8808


xlm-r test: typed micro F1=0.7976, macro F1=0.7131, boundary F1=0.8496
Best model: /content/drive/MyDrive/Persian_NER_Revised/run_20260929T190017_726808Z/xlm-r/best_model


In [6]:
result_parsbert = run_one_model("parsbert")


Saving the dataset (0/1 shards):   0%|          | 0/4598 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/512 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2560 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4598 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/512 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

parsbert validation: typed micro F1=0.8235, macro F1=0.7519, boundary F1=0.8717


parsbert test: typed micro F1=0.8076, macro F1=0.7338, boundary F1=0.8558
Best model: /content/drive/MyDrive/Persian_NER_Revised/run_20260929T190017_726808Z/parsbert/best_model


In [7]:
result_tookabert = run_one_model("tookabert")


Saving the dataset (0/1 shards):   0%|          | 0/4598 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/512 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2560 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4598 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/512 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

tookabert validation: typed micro F1=0.8529, macro F1=0.7923, boundary F1=0.8894


tookabert test: typed micro F1=0.8111, macro F1=0.7285, boundary F1=0.8606
Best model: /content/drive/MyDrive/Persian_NER_Revised/run_20260929T190017_726808Z/tookabert/best_model


In [8]:
print("Run folder:", REVISED_RUN_DIR)
print(f"{'Model':<12} {'Typed micro F1':>16} {'Macro F1':>12} {'Boundary F1':>14}")
for key in MODEL_ORDER:
    if key not in test_predictions_store:
        print(f"{key:<12} Not completed in this run")
        continue
    scores = test_predictions_store[key]["test_scores"]
    print(f"{key:<12} {scores['typed_micro']['f1']:>16.4f} "
          f"{scores['typed_macro']['f1']:>12.4f} "
          f"{scores['boundary_micro']['f1']:>14.4f}")
print("Test scores from checkpoints selected using validation data.")
print("Fragmentation and boundary-error association analyses remain separate.")


Run folder: /content/drive/MyDrive/Persian_NER_Revised/run_20260929T190017_726808Z
Model          Typed micro F1     Macro F1    Boundary F1
xlm-r                  0.7976       0.7131         0.8496
parsbert               0.8076       0.7338         0.8558
tookabert              0.8111       0.7285         0.8606
Test scores from checkpoints selected using validation data.
Fragmentation and boundary-error association analyses remain separate.


In [9]:
import csv
from pathlib import Path

if "REVISED_RUN_DIR" not in globals():
    raise RuntimeError(
        "REVISED_RUN_DIR is missing. If Colab restarted, set it to the "
        "existing run folder in Google Drive. Do not rerun training."
    )

run_dir = Path(REVISED_RUN_DIR)

models = ["xlm-r", "parsbert", "tookabert"]
entity_classes = ["PER", "ORG", "LOC", "FAC", "EVENT", "PROD"]

per_class_results = {}


for model_name in models:
    file_path = run_dir / model_name / "test_per_class.csv"

    if not file_path.is_file():
        raise FileNotFoundError(
            f"Missing results for {model_name}:\n{file_path}\n"

        )

    with file_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)

        required_columns = {"class", "precision", "recall", "f1", "support"}
        if not required_columns.issubset(set(reader.fieldnames or [])):
            raise ValueError(f"Unexpected CSV columns in {file_path}")

        rows = list(reader)

    if (
        len(rows) != len(entity_classes)
        or {row["class"] for row in rows} != set(entity_classes)
    ):
        raise ValueError(
            f"{model_name}: expected exactly one row for each of the six classes."
        )

    model_results = {}

    for row in rows:
        entity_type = row["class"]

        values = {
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "f1": float(row["f1"]),
            "support": int(row["support"]),
        }

        for metric in ["precision", "recall", "f1"]:
            if not 0.0 <= values[metric] <= 1.0:
                raise ValueError(
                    f"{model_name}, {entity_type}: invalid {metric}."
                )

        if values["support"] < 0:
            raise ValueError(f"{model_name}, {entity_type}: negative support.")

        model_results[entity_type] = values

    per_class_results[model_name] = model_results

for entity_type in entity_classes:
    supports = {
        per_class_results[model_name][entity_type]["support"]
        for model_name in models
    }

    if len(supports) != 1:
        raise ValueError(
            f"Gold support differs between models for {entity_type}. "
            "Check that all files belong to the same corrected test split."
        )

print("Run folder:", run_dir)
print("Scores use the 0–1 scale.")
print("Support = gold entity mentions.")

for model_name in models:
    print("\n" + "=" * 70)
    print(model_name.upper())
    print("=" * 70)
    print(
        f"{'Class':<10}"
        f"{'Precision':>14}"
        f"{'Recall':>14}"
        f"{'F1':>14}"
        f"{'Support':>12}"
    )

    for entity_type in entity_classes:
        values = per_class_results[model_name][entity_type]
        print(
            f"{entity_type:<10}"
            f"{values['precision']:>14.4f}"
            f"{values['recall']:>14.4f}"
            f"{values['f1']:>14.4f}"
            f"{values['support']:>12,d}"
        )

    macro_f1 = sum(
        per_class_results[model_name][entity_type]["f1"]
        for entity_type in entity_classes
    ) / len(entity_classes)

    print(f"\nMacro F1 across all six classes: {macro_f1:.4f}")

print("\n" + "=" * 70)
print("PER-CLASS TYPED ENTITY F1 COMPARISON")
print("=" * 70)
print(
    f"{'Class':<10}"
    f"{'XLM-R':>14}"
    f"{'ParsBERT':>14}"
    f"{'TookaBERT':>14}"
    f"{'Support':>12}"
)

for entity_type in entity_classes:
    print(
        f"{entity_type:<10}"
        f"{per_class_results['xlm-r'][entity_type]['f1']:>14.4f}"
        f"{per_class_results['parsbert'][entity_type]['f1']:>14.4f}"
        f"{per_class_results['tookabert'][entity_type]['f1']:>14.4f}"
        f"{per_class_results['xlm-r'][entity_type]['support']:>12,d}"
    )


print("\n" + "=" * 70)
print("TOOKABERT MINUS PARSBERT: F1 DIFFERENCES")
print("Units: percentage points. Positive = higher TookaBERT F1.")
print("=" * 70)

differences = {}

for entity_type in entity_classes:
    difference = 100 * (
        per_class_results["tookabert"][entity_type]["f1"]
        - per_class_results["parsbert"][entity_type]["f1"]
    )
    differences[entity_type] = difference
    print(f"{entity_type:<10}{difference:>+12.2f}")

print(
    "\nMacro F1 difference: "
    f"{sum(differences.values()) / len(entity_classes):+.2f} percentage points"
)

print("\nInspection complete. No data or model weights were changed.")

Run folder: /content/drive/MyDrive/Persian_NER_Revised/run_20260929T190017_726808Z
Scores use the 0–1 scale.
Support = gold entity mentions.

XLM-R
Class          Precision        Recall            F1     Support
PER               0.9150        0.9315        0.9232       1,110
ORG               0.7299        0.8424        0.7821       1,466
LOC               0.8275        0.7898        0.8082       1,142
FAC               0.6010        0.7126        0.6521         167
EVENT             0.4805        0.6271        0.5441         177
PROD              0.7251        0.4679        0.5688         265

Macro F1 across all six classes: 0.7131

PARSBERT
Class          Precision        Recall            F1     Support
PER               0.9176        0.9532        0.9350       1,110
ORG               0.7619        0.8056        0.7832       1,466
LOC               0.7957        0.8494        0.8217       1,142
FAC               0.7029        0.7365        0.7193         167
EVENT             0.4

In [10]:

import csv
import json
import hashlib
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

MODELS = ["xlm-r", "parsbert", "tookabert"]
CLASSES = ["PER", "ORG", "LOC", "FAC", "EVENT", "PROD"]

if "REVISED_RUN_DIR" not in globals():
    raise RuntimeError("Set REVISED_RUN_DIR to your existing saved training run folder first.")
run_dir = Path(REVISED_RUN_DIR)

def diagnostic_spans(tags, strict_gold=False):
    spans = set()
    start, active = None, None
    for i, tag in enumerate(list(tags) + ["O"]):
        if tag == "O":
            prefix, kind = "O", None
        else:
            prefix, kind = tag.split("-", 1)
            if prefix not in ("B", "I") or kind not in CLASSES:
                raise ValueError(f"Unknown label: {tag}")
        continues = prefix == "I" and active == kind
        if prefix == "I" and not continues and strict_gold:
            raise ValueError(f"Invalid gold BIO at word {i}: {tag}")
        if active is not None and not continues:
            spans.add((start, i, active))
            start, active = None, None
        if prefix != "O" and not continues:
            start, active = i, kind
    return spans


def read_jsonl(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def write_csv(path, rows):
    if not rows:
        return
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def average(values):
    return sum(values) / len(values) if values else None


def frequency_band(n):
    return "0" if n == 0 else "1" if n == 1 else "2-5" if n <= 5 else "6+"


def display(value, digits=4):
    return "NA" if value is None else f"{value:.{digits}f}"

from datasets import load_from_disk
saved_splits = load_from_disk(str(run_dir / "clean_splits"))
with (run_dir / "label_map.json").open(encoding="utf-8") as handle:
    saved_labels = json.load(handle)["labels"]
train_frequency = Counter()
for sentence in saved_splits["train"]:
    tags = [saved_labels[int(i)] for i in sentence["ner_tags"]]
    for start, end, kind in diagnostic_spans(tags, strict_gold=True):
        train_frequency[(kind, tuple(sentence["tokens"][start:end]))] += 1

test_gold = {
    row["sentence_id"]: (list(row["tokens"]), [saved_labels[int(i)] for i in row["ner_tags"]])
    for row in saved_splits["test"]
}
assert len(test_gold) == len(saved_splits["test"]), "Duplicate test sentence IDs."

mention_rows = []
word_stats = defaultdict(Counter)
source_hashes = {}
for model in MODELS:
    path = run_dir / model / "test_word_predictions.jsonl"
    if not path.is_file():
        raise FileNotFoundError(path)
    source_hashes[model] = hashlib.sha256(path.read_bytes()).hexdigest()
    records = read_jsonl(path)
    seen = set()
    for record in records:
        sid = record["sentence_id"]
        if sid in seen or sid not in test_gold:
            raise ValueError(f"{model}: duplicate or unknown sentence ID {sid}")
        seen.add(sid)
        words = record["tokens"]
        gold, predicted = record["gold_labels"], record["predicted_labels"]
        counts = record["input_subwords_per_word"]
        fallback = record["fallback_word_mask"]
        if (words, gold) != test_gold[sid]:
            raise ValueError(f"{model}: gold text/labels differ in {sid}")
        if not len(words) == len(gold) == len(predicted) == len(counts) == len(fallback):
            raise ValueError(f"{model}: incomplete coverage in {sid}")
        if any(not isinstance(n, int) or n < 1 for n in counts):
            raise ValueError(f"{model}: invalid subword counts in {sid}")
        if any(not isinstance(f, bool) for f in fallback):
            raise ValueError(f"{model}: invalid fallback flags in {sid}")
        if any(f and n != 1 for n, f in zip(counts, fallback)):
            raise ValueError("Expected exactly one inserted UNK for each fallback word.")
        gs = diagnostic_spans(gold, strict_gold=True)
        ps = diagnostic_spans(predicted)
        predicted_by_boundary = {(s, e): t for s, e, t in ps}
        for tag, pieces, is_fallback in zip(gold, counts, fallback):
            scopes = ["ALL_WORDS"]
            if tag != "O":
                scopes += ["ENTITY_WORDS", tag.split("-", 1)[1]]
            for scope in scopes:
                stats = word_stats[(model, scope)]
                stats["original_words"] += 1
                stats["fallback_words"] += int(is_fallback)
                stats["input_subwords"] += pieces
                if not is_fallback:
                    stats["native_represented_words"] += 1
                    stats["native_subwords"] += pieces
                    stats["fragmented_words"] += int(pieces > 1)
        for start, end, kind in sorted(gs):
            length = end - start
            n_fallback = sum(fallback[start:end])
            input_pieces = sum(counts[start:end])
            frequency = train_frequency[(kind, tuple(words[start:end]))]
            boundary_correct = (start, end) in predicted_by_boundary
            typed_correct = (start, end, kind) in ps
            native_sfr = input_pieces / length if n_fallback == 0 else None
            mention_rows.append({
                "model": model, "sentence_id": sid, "start": start, "end": end,
                "class": kind, "text": " ".join(words[start:end]),
                "entity_words": length, "train_entity_frequency": frequency,
                "frequency_band": frequency_band(frequency),
                "input_subwords": input_pieces,
                "native_subwords": input_pieces - n_fallback,
                "fallback_words": n_fallback,
                "native_sfr_no_fallback": native_sfr,
                "fragmented": int(input_pieces > length) if n_fallback == 0 else None,
                "boundary_correct": int(boundary_correct),
                "boundary_error": int(not boundary_correct),
                "typed_correct": int(typed_correct),
                "predicted_type_at_exact_boundary": predicted_by_boundary.get((start, end), ""),
                "type_correct_given_boundary": int(typed_correct) if boundary_correct else None,
            })
    if seen != set(test_gold):
        raise ValueError(f"{model}: some test sentences have no saved predictions.")

entity_sets = {
    model: {(r["sentence_id"], r["start"], r["end"], r["class"])
            for r in mention_rows if r["model"] == model}
    for model in MODELS
}
assert all(entity_sets[m] == entity_sets[MODELS[0]] for m in MODELS)

fragmentation_rows = []
for (model, scope), stats in word_stats.items():
    represented = stats["native_represented_words"]
    fragmentation_rows.append({
        "model": model, "scope": scope, **dict(stats),
        "native_subwords_per_represented_word": stats["native_subwords"] / represented if represented else None,
        "fragmented_fraction_of_represented_words": stats["fragmented_words"] / represented if represented else None,
    })

class_rows = []
for model in MODELS:
    for scope in ["ALL_ENTITIES"] + CLASSES:
        rows = [r for r in mention_rows if r["model"] == model and
                (scope == "ALL_ENTITIES" or r["class"] == scope)]
        matched = [r for r in rows if r["boundary_correct"]]
        eligible = [r for r in rows if r["fallback_words"] == 0]
        class_rows.append({
            "model": model, "class": scope, "gold_mentions": len(rows),
            "boundary_recall": average([r["boundary_correct"] for r in rows]),
            "typed_recall": average([r["typed_correct"] for r in rows]),
            "boundary_matched_mentions": len(matched),
            "typing_accuracy_given_boundary": average([r["typed_correct"] for r in matched]),
            "mentions_with_fallback": len(rows) - len(eligible),
            "native_sfr_eligible_mentions": len(eligible),
            "mean_mention_native_sfr": average([r["native_sfr_no_fallback"] for r in eligible]),
        })

fallback_entity_ids = {
    (r["sentence_id"], r["start"], r["end"], r["class"])
    for r in mention_rows if r["fallback_words"] > 0
}
main_rows = [r for r in mention_rows if
             (r["sentence_id"], r["start"], r["end"], r["class"]) not in fallback_entity_ids]

def group_summary(model, scope, label, rows):
    return {"model": model, "scope": scope, "fragmentation_group": label,
            "gold_mentions": len(rows),
            "boundary_error_rate": average([r["boundary_error"] for r in rows]),
            "boundary_recall": average([r["boundary_correct"] for r in rows]),
            "typed_recall": average([r["typed_correct"] for r in rows]),
            "mean_entity_words": average([r["entity_words"] for r in rows])}

group_rows, stratum_rows, adjusted_rows = [], [], []
for model in MODELS:
    for scope in ["ALL_ENTITIES"] + CLASSES:
        rows = [r for r in main_rows if r["model"] == model and
                (scope == "ALL_ENTITIES" or r["class"] == scope)]
        for fragmented, label in [(0, "one_piece_per_word"), (1, "some_words_split")]:
            selected = [r for r in rows if r["fragmented"] == fragmented]
            group_rows.append(group_summary(model, scope, label, selected))

        strata = defaultdict(lambda: {0: [], 1: []})
        for r in rows:
            key = (r["class"], r["entity_words"], r["frequency_band"])
            strata[key][r["fragmented"]].append(r)
        weighted_difference, matched_n, matched_strata = 0.0, 0, 0
        for (kind, length, band), groups in sorted(strata.items()):
            a, b = groups[0], groups[1]
            error_a = average([r["boundary_error"] for r in a])
            error_b = average([r["boundary_error"] for r in b])
            difference = error_b - error_a if a and b else None
            stratum_rows.append({
                "model": model, "scope": scope, "class": kind,
                "entity_words": length, "frequency_band": band,
                "unfragmented_n": len(a), "fragmented_n": len(b),
                "unfragmented_error_rate": error_a, "fragmented_error_rate": error_b,
                "difference_fragmented_minus_unfragmented": difference,
            })
            if difference is not None:
                weight = len(a) + len(b)
                weighted_difference += weight * difference
                matched_n += weight
                matched_strata += 1
        adjusted_rows.append({
            "model": model, "scope": scope, "eligible_mentions": len(rows),
            "mentions_in_comparable_strata": matched_n,
            "comparable_strata": matched_strata,
            "comparable_coverage": matched_n / len(rows) if rows else None,
            "weighted_boundary_error_difference": weighted_difference / matched_n if matched_n else None,
        })


analysis_dir = run_dir / datetime.now(timezone.utc).strftime("fragmentation_%Y%m%dT%H%M%S_%fZ")
analysis_dir.mkdir(parents=True, exist_ok=False)
for name, rows in [
    ("entity_diagnostics", mention_rows), ("native_word_fragmentation", fragmentation_rows),
    ("class_boundary_summary", class_rows), ("fragmentation_groups", group_rows),
    ("length_frequency_strata", stratum_rows), ("adjusted_descriptive_comparison", adjusted_rows),
]:
    write_csv(analysis_dir / f"{name}.csv", rows)
with (analysis_dir / "analysis_policy.json").open("w", encoding="utf-8") as handle:
    json.dump({
        "prediction_file_sha256": source_hashes,
        "gold_test_sentences": len(test_gold),
        "gold_mentions_per_model": len(entity_sets[MODELS[0]]),
        "entities_excluded_from_main_comparison_for_any_model_fallback": len(fallback_entity_ids),
        "fragmentation": "Subwords per original word; >1 means fragmented. Native word summaries omit zero-output words from the denominator and report their fallback counts separately.",
        "main_comparison": "Common gold entities without fallback in ANY model; entity fragmented if any word has >1 pieces.",
        "frequency": "Exact token-sequence AND entity-type counts among gold training entities; bands 0, 1, 2-5, 6+.",
        "adjustment": "Within-class, exact-length, frequency-band differences; only strata containing both groups; weighted by pooled stratum size. Descriptive only, no confidence intervals or causal inference.",
        "boundary_recall": "Fraction of gold entities recovered at exact start/end, regardless of predicted type; not boundary F1.",
        "predicted_BIO": "An incompatible I-TYPE starts a new span.",
        "scope_limit": "Does not classify false-positive spans or demonstrate reduced reliance on capitalization. Native unknown-token emissions cannot be identified separately from these saved fields.",
    }, handle, ensure_ascii=False, indent=2)

# 7. Compact output to share for interpretation.
print("All three prediction files match the saved gold test split.")
print("Gold entity mentions per model:", len(entity_sets[MODELS[0]]))
print("Gold entities excluded from common fragmentation comparison:", len(fallback_entity_ids))
print("\nNATIVE TOKENIZATION — ALL ORIGINAL WORDS")
print("Mean pieces uses represented words only; fallback words are separate.")
for row in fragmentation_rows:
    if row["scope"] == "ALL_WORDS":
        print(f"{row['model']:<10} mean pieces={display(row['native_subwords_per_represented_word'])} "
              f"represented={row.get('native_represented_words', 0):,} "
              f"fallback={row.get('fallback_words', 0):,}")
print("\nPER: BOUNDARY RECOVERY AND TYPING — ALL GOLD PER MENTIONS")
for row in class_rows:
    if row["class"] == "PER":
        print(f"{row['model']:<10} n={row['gold_mentions']} "
              f"boundary recall={display(row['boundary_recall'])} "
              f"typed recall={display(row['typed_recall'])} "
              f"typing given boundary={display(row['typing_accuracy_given_boundary'])} "
              f"mean mention SFR={display(row['mean_mention_native_sfr'])} "
              f"fallback mentions={row['mentions_with_fallback']}")
print("\nPER: UNADJUSTED FRAGMENTATION GROUPS — COMMON NO-FALLBACK SUBSET")
for row in group_rows:
    if row["scope"] == "PER":
        print(f"{row['model']:<10} {row['fragmentation_group']:<20} "
              f"n={row['gold_mentions']:<5} boundary error={display(row['boundary_error_rate'])} "
              f"mean words={display(row['mean_entity_words'], 2)}")
print("\nPER: WITHIN-LENGTH/FREQUENCY COMPARISON — DESCRIPTIVE ONLY")
print("Positive difference = higher boundary error for fragmented mentions.")
for row in adjusted_rows:
    if row["scope"] == "PER":
        difference = row['weighted_boundary_error_difference']
        print(f"{row['model']:<10} difference(pp)={display(None if difference is None else 100*difference, 2)} "
              f"comparable mentions={row['mentions_in_comparable_strata']}/{row['eligible_mentions']} "
              f"strata={row['comparable_strata']}")
print("\nSaved analysis:", analysis_dir)
print("No training or inference performed. Share the printed output for interpretation.")


All three prediction files match the saved gold test split.
Gold entity mentions per model: 4327
Gold entities excluded from common fragmentation comparison: 9

NATIVE TOKENIZATION — ALL ORIGINAL WORDS
Mean pieces uses represented words only; fallback words are separate.
xlm-r      mean pieces=1.2886 represented=82,119 fallback=0
parsbert   mean pieces=1.0225 represented=82,119 fallback=0
tookabert  mean pieces=1.1033 represented=81,799 fallback=320

PER: BOUNDARY RECOVERY AND TYPING — ALL GOLD PER MENTIONS
xlm-r      n=1110 boundary recall=0.9396 typed recall=0.9315 typing given boundary=0.9914 mean mention SFR=2.0808 fallback mentions=0
parsbert   n=1110 boundary recall=0.9631 typed recall=0.9532 typing given boundary=0.9897 mean mention SFR=1.1609 fallback mentions=0
tookabert  n=1110 boundary recall=0.9505 typed recall=0.9441 typing given boundary=0.9934 mean mention SFR=1.3294 fallback mentions=0

PER: UNADJUSTED FRAGMENTATION GROUPS — COMMON NO-FALLBACK SUBSET
xlm-r      one_piec

In [11]:
import csv
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import numpy as np

if 'REVISED_RUN_DIR' not in globals() or 'analysis_dir' not in globals():
    raise RuntimeError('set REVISED_RUN_DIR and analysis_dir to the existing saved folders.')
followup_run = Path(REVISED_RUN_DIR)
followup_analysis = Path(analysis_dir)
if followup_analysis.parent.resolve() != followup_run.resolve():
    raise ValueError('The fragmentation folder does not belong to this training run.')
followup_models = ['xlm-r', 'parsbert', 'tookabert']
followup_classes = ['PER', 'ORG', 'LOC', 'FAC', 'EVENT', 'PROD']

def followup_csv(path):
    with Path(path).open(encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))

def followup_write(path, rows):
    if rows:
        with Path(path).open('w', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)

def followup_spans(tags, strict=False):
    spans = set()
    start, active = None, None
    for i, tag in enumerate(list(tags) + ['O']):
        if tag == 'O':
            prefix, kind = 'O', None
        else:
            prefix, kind = tag.split('-', 1)
            if prefix not in ('B', 'I') or kind not in followup_classes:
                raise ValueError(f'Invalid BIO label {tag}')
        continues = prefix == 'I' and active == kind
        if strict and prefix == 'I' and not continues:
            raise ValueError('Invalid gold BIO sequence.')
        if active is not None and not continues:
            spans.add((start, i, active))
            start, active = None, None
        if prefix != 'O' and not continues:
            start, active = i, kind
    return spans

with (followup_analysis / 'analysis_policy.json').open(encoding='utf-8') as f:
    followup_policy = json.load(f)
followup_records = {}
followup_hashes = {}
for model in followup_models:
    path = followup_run / model / 'test_word_predictions.jsonl'
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != followup_policy['prediction_file_sha256'][model]:
        raise ValueError(f'{model}: predictions changed since Cell 4. Re-run Cell 4 on the intended files.')
    followup_hashes[model] = digest
    with path.open(encoding='utf-8') as f:
        records = [json.loads(line) for line in f if line.strip()]
    lookup = {r['sentence_id']: r for r in records}
    if len(lookup) != len(records):
        raise ValueError('Duplicate sentence IDs.')
    followup_records[model] = lookup
followup_ids = list(followup_records['xlm-r'])
for model in followup_models:
    if set(followup_records[model]) != set(followup_ids):
        raise ValueError('Models do not have the same test sentences.')
    for sid in followup_ids:
        r, ref = followup_records[model][sid], followup_records['xlm-r'][sid]
        if r['tokens'] != ref['tokens'] or r['gold_labels'] != ref['gold_labels']:
            raise ValueError(f'Gold data mismatch: {model}, {sid}')
        if not len(r['tokens']) == len(r['gold_labels']) == len(r['predicted_labels']):
            raise ValueError('Incomplete word coverage.')

all_mentions = followup_csv(followup_analysis / 'entity_diagnostics.csv')
def followup_key(r):
    return (r['sentence_id'], int(r['start']), int(r['end']), r['class'])
excluded = {followup_key(r) for r in all_mentions if int(r['fallback_words']) > 0}
xlm_per = [r for r in all_mentions if r['model'] == 'xlm-r' and r['class'] == 'PER'
           and followup_key(r) not in excluded]
unfragmented = [r for r in xlm_per if int(r['fragmented']) == 0]
fragmented = [r for r in xlm_per if int(r['fragmented']) == 1]
errors = [r for r in unfragmented if int(r['boundary_error']) == 1]
print('XLM-R PER boundary-error counts (gold mentions, not false-positive predictions)')
for label, rows in [('Unfragmented', unfragmented), ('Fragmented', fragmented)]:
    n_error = sum(int(r['boundary_error']) for r in rows)
    print(f'{label}: {n_error}/{len(rows)} errors; rate={n_error / len(rows):.4f}' if rows else f'{label}: no mentions')

name_groups = defaultdict(list)
for r in unfragmented:
    source = followup_records['xlm-r'][r['sentence_id']]
    name_groups[tuple(source['tokens'][int(r['start']):int(r['end'])])].append(r)
name_rows = []
for tokens, rows in name_groups.items():
    n_error = sum(int(r['boundary_error']) for r in rows)
    name_rows.append({'name': ' '.join(tokens), 'token_tuple_json': json.dumps(tokens, ensure_ascii=False),
                     'test_mentions': len(rows), 'boundary_errors': n_error,
                     'boundary_error_rate': n_error / len(rows),
                     'train_entity_frequency': int(rows[0]['train_entity_frequency']),
                     'distinct_error_sentences': len({r['sentence_id'] for r in rows if int(r['boundary_error'])})})
name_rows.sort(key=lambda r: (-r['boundary_errors'], -r['test_mentions'], r['name']))
error_names = [r for r in name_rows if r['boundary_errors']]
print('\nNames contributing the most unfragmented errors (top 10):')
for r in error_names[:10]:
    print(f"{r['name']} | errors={r['boundary_errors']}/{r['test_mentions']} "
          f"| train frequency={r['train_entity_frequency']}")
for k in (1, 3, 5):
    count = sum(r['boundary_errors'] for r in error_names[:k])
    print(f'Top {k} names account for {count}/{len(errors)} errors.')

name_rank = {r['token_tuple_json']: i for i, r in enumerate(error_names)}
case_rows = []
for r in errors:
    sid, start, end = r['sentence_id'], int(r['start']), int(r['end'])
    source = followup_records['xlm-r'][sid]
    words = source['tokens']
    left, right = max(0, start - 6), min(len(words), end + 6)
    context = ' '.join(words[left:start]) + ' [' + ' '.join(words[start:end]) + '] ' + ' '.join(words[end:right])
    case = {'sentence_id': sid, 'start': start, 'end': end,
            'name': ' '.join(words[start:end]), 'context': context.strip(),
            'entity_words': end - start, 'train_entity_frequency': int(r['train_entity_frequency']),
            'gold_context_labels': ' '.join(source['gold_labels'][left:right]),
            'xlmr_context_labels': ' '.join(source['predicted_labels'][left:right])}
    for model in followup_models:
        predicted_spans = followup_spans(followup_records[model][sid]['predicted_labels'])
        overlaps = [{'start': s, 'end': e, 'type': t, 'text': ' '.join(words[s:e])}
                    for s, e, t in sorted(predicted_spans) if s < end and e > start]
        case[f'{model}_overlapping_predictions'] = json.dumps(overlaps, ensure_ascii=False)
        case[f'{model}_exact_boundary'] = int(any(s == start and e == end for s, e, t in predicted_spans))
        case[f'{model}_exact_PER'] = int((start, end, 'PER') in predicted_spans)
    case['_rank'] = name_rank[json.dumps(tuple(words[start:end]), ensure_ascii=False)]
    case_rows.append(case)
case_rows.sort(key=lambda r: (r['_rank'], r['sentence_id'], r['start']))
for r in case_rows:
    del r['_rank']
print('\nExample contexts (brackets mark the gold PER; all cases saved):')
for r in case_rows[:10]:
    print(f"\n{r['sentence_id']} | {r['context']}")
    print('XLM-R overlapping predictions:', r['xlm-r_overlapping_predictions'])
    print('Exact boundary recovered: ParsBERT=', r['parsbert_exact_boundary'],
          'TookaBERT=', r['tookabert_exact_boundary'])

strata = [r for r in followup_csv(followup_analysis / 'length_frequency_strata.csv')
          if r['model'] == 'xlm-r' and r['scope'] == 'PER']
comparable_n = sum(int(r['unfragmented_n']) + int(r['fragmented_n']) for r in strata
                   if int(r['unfragmented_n']) > 0 and int(r['fragmented_n']) > 0)
influence_rows = []
for r in strata:
    n0, n1 = int(r['unfragmented_n']), int(r['fragmented_n'])
    comparable = n0 > 0 and n1 > 0
    difference = float(r['difference_fragmented_minus_unfragmented']) if comparable else None
    weight = (n0 + n1) / comparable_n if comparable and comparable_n else 0.0
    influence_rows.append({**r, 'weight_in_adjusted_difference': weight,
                           'contribution_percentage_points': 100 * weight * difference if comparable else None,
                           'either_group_below_10': int(min(n0, n1) < 10)})
influence_rows.sort(key=lambda r: -abs(r['contribution_percentage_points'] or 0))
print('\nXLM-R PER strata: largest contributions to adjusted difference (top 10)')
print('Small group flag (<10) is an inspection aid, not a statistical threshold.')
for r in influence_rows[:10]:
    contribution = r['contribution_percentage_points']
    print(f"words={r['entity_words']} frequency={r['frequency_band']} "
          f"n(unfrag/frag)={r['unfragmented_n']}/{r['fragmented_n']} "
          f"error(unfrag/frag)={r['unfragmented_error_rate']}/{r['fragmented_error_rate']} "
          f"contribution(pp)={None if contribution is None else round(contribution, 2)} "
          f"small_group={r['either_group_below_10']}")
print('Sum of contributions (pp):', round(sum(r['contribution_percentage_points'] or 0 for r in influence_rows), 2))

followup_dir = followup_run / datetime.now(timezone.utc).strftime('followup_%Y%m%dT%H%M%S_%fZ')
followup_dir.mkdir(exist_ok=False)
followup_write(followup_dir / 'xlmr_unfragmented_PER_names.csv', name_rows)
followup_write(followup_dir / 'xlmr_unfragmented_PER_errors.csv', case_rows)
followup_write(followup_dir / 'xlmr_PER_stratum_influence.csv', influence_rows)
with (followup_dir / 'sources.json').open('w', encoding='utf-8') as f:
    json.dump({'run': str(followup_run), 'fragmentation_analysis': str(followup_analysis),
               'prediction_sha256': followup_hashes}, f, indent=2)
followup_ready = True
print('\nInspection saved to:', followup_dir)
print('No causes assigned automatically. Inspect the contexts before interpreting the pattern.')


XLM-R PER boundary-error counts (gold mentions, not false-positive predictions)
Unfragmented: 25/159 errors; rate=0.1572
Fragmented: 42/951 errors; rate=0.0442

Names contributing the most unfragmented errors (top 10):
امام خمینی | errors=3/15 | train frequency=25
حسین | errors=2/3 | train frequency=2
خمینی | errors=1/6 | train frequency=6
فردوسی | errors=1/4 | train frequency=1
مولوی | errors=1/4 | train frequency=3
ال گور | errors=1/2 | train frequency=4
استاد | errors=1/1 | train frequency=0
امام جواد | errors=1/1 | train frequency=0
امام زمان | errors=1/1 | train frequency=0
امام موسی | errors=1/1 | train frequency=0
Top 1 names account for 3/25 errors.
Top 3 names account for 6/25 errors.
Top 5 names account for 8/25 errors.

Example contexts (brackets mark the gold PER; all cases saved):

testfile_01964 | مصطفوی روز پنج‌شنبه در حرم مطهر [امام خمینی] » ره « گفت : امام
XLM-R overlapping predictions: [{"start": 9, "end": 13, "type": "FAC", "text": "حرم مطهر امام خمینی"}]
Exact bound

In [12]:
if not globals().get('followup_ready', False):
    raise RuntimeError('Run Cell 5A first in this runtime.')
BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 2026
bootstrap_metrics = ['typed_micro_f1', 'typed_macro_f1_six_classes',
                     'boundary_micro_f1', 'PER_typed_f1', 'PER_boundary_recall']


def followup_sentence_counts(record):
    gold = followup_spans(record['gold_labels'], strict=True)
    predicted = followup_spans(record['predicted_labels'])
    matched = gold & predicted
    counts = np.zeros(20, dtype=np.int64)
    for j, kind in enumerate(followup_classes):
        counts[j] = sum(t == kind for s, e, t in matched)
        counts[6+j] = sum(t == kind for s, e, t in predicted)
        counts[12+j] = sum(t == kind for s, e, t in gold)
    gb = {(s, e) for s, e, t in gold}
    pb = {(s, e) for s, e, t in predicted}
    counts[18] = len(gb & pb)
    counts[19] = sum((s, e) in pb for s, e, t in gold if t == 'PER')
    return counts


def followup_metrics_from_counts(counts):
    counts = np.asarray(counts, dtype=np.float64)
    tp, predicted, gold = counts[..., :6], counts[..., 6:12], counts[..., 12:18]
    def ratio(numerator, denominator):
        return np.divide(numerator, denominator, out=np.zeros_like(numerator, dtype=float), where=denominator != 0)
    class_f1 = ratio(2 * tp, predicted + gold)
    total_denominator = predicted.sum(axis=-1) + gold.sum(axis=-1)
    return np.stack([
        ratio(2 * tp.sum(axis=-1), total_denominator),
        class_f1.mean(axis=-1),
        ratio(2 * counts[..., 18], total_denominator),
        class_f1[..., 0],
        ratio(counts[..., 19], gold[..., 0]),
    ], axis=-1)

sentence_counts = np.stack([
    np.stack([followup_sentence_counts(followup_records[model][sid]) for sid in followup_ids])
    for model in followup_models
])
assert np.all(sentence_counts[:, :, 12:18] == sentence_counts[0, :, 12:18])
point_estimates = followup_metrics_from_counts(sentence_counts.sum(axis=1))
# Check exact agreement with Cell 2 and Cell 4 before resampling.
class_summary = followup_csv(followup_analysis / 'class_boundary_summary.csv')
for i, model in enumerate(followup_models):
    with (followup_run / model / 'test_scores.json').open(encoding='utf-8') as f:
        saved = json.load(f)
    per_boundary = next(float(r['boundary_recall']) for r in class_summary
                        if r['model'] == model and r['class'] == 'PER')
    expected = [saved['typed_micro']['f1'], saved['typed_macro']['f1'],
                saved['boundary_micro']['f1'], saved['per_class']['PER']['f1'], per_boundary]
    if not np.allclose(point_estimates[i], expected, atol=1e-10, rtol=0):
        raise ValueError(f'{model}: recomputed metrics differ from saved scores. Stop and inspect.')
print('All five recomputed metrics match saved scores for all three models.')

rng = np.random.default_rng(BOOTSTRAP_SEED)
n_sentences = len(followup_ids)
bootstrap_scores = np.empty((BOOTSTRAP_REPLICATES, len(followup_models), len(bootstrap_metrics)))

for first in range(0, BOOTSTRAP_REPLICATES, 50):
    size = min(50, BOOTSTRAP_REPLICATES - first)
    ids = rng.integers(0, n_sentences, size=(size, n_sentences))
    weights = np.stack([np.bincount(row, minlength=n_sentences) for row in ids])
    pooled = np.einsum('bn,mnk->bmk', weights, sentence_counts, optimize=False)
    bootstrap_scores[first:first+size] = followup_metrics_from_counts(pooled)
    if (first + size) % 1000 == 0 or first + size == BOOTSTRAP_REPLICATES:
        print(f'Bootstrap: {first + size}/{BOOTSTRAP_REPLICATES}')

pair_rows, model_rows = [], []
for a, b in [('parsbert', 'xlm-r'), ('tookabert', 'xlm-r'), ('tookabert', 'parsbert')]:
    ia, ib = followup_models.index(a), followup_models.index(b)
    for j, metric in enumerate(bootstrap_metrics):
        distribution = 100 * (bootstrap_scores[:, ia, j] - bootstrap_scores[:, ib, j])
        low, high = np.quantile(distribution, [0.025, 0.975])
        pair_rows.append({'first_model': a, 'second_model': b, 'metric': metric,
                          'difference_percentage_points': float(100 * (point_estimates[ia, j] - point_estimates[ib, j])),
                          'ci95_low_pp': float(low), 'ci95_high_pp': float(high),
                          'interval_contains_zero': int(low <= 0 <= high)})
for i, model in enumerate(followup_models):
    for j, metric in enumerate(bootstrap_metrics):
        low, high = np.quantile(bootstrap_scores[:, i, j], [0.025, 0.975])
        model_rows.append({'model': model, 'metric': metric, 'score': float(point_estimates[i, j]),
                           'ci95_low': float(low), 'ci95_high': float(high)})
followup_write(followup_dir / 'paired_bootstrap_differences.csv', pair_rows)
followup_write(followup_dir / 'bootstrap_model_scores.csv', model_rows)
np.savez_compressed(followup_dir / 'bootstrap_samples.npz', scores=bootstrap_scores,
                    models=np.array(followup_models), metrics=np.array(bootstrap_metrics),
                    sentence_ids=np.array(followup_ids), sentence_counts=sentence_counts)
with (followup_dir / 'bootstrap_policy.json').open('w', encoding='utf-8') as f:
    json.dump({'replicates': BOOTSTRAP_REPLICATES, 'seed': BOOTSTRAP_SEED,
               'numpy_version': np.__version__, 'unit': 'sentence', 'n_sentences': n_sentences,
               'interval': '95% paired percentile bootstrap; marginal, no multiple-comparison adjustment',
               'pairing': 'Identical resampled sentence IDs for all models in each replicate',
               'aggregation': 'Recompute corpus metrics from summed counts, never average sentence F1',
               'zero_denominator': 'Metric set to zero, matching corrected evaluator',
               'population': 'All original saved test sentences, including fallback and entity-free sentences',
               'limits': 'Conditional on these saved single-seed models. Does not measure retraining variability, prove equivalence, or establish causality. Sentence sampling assumes independent sentences; document/topic dependencies can make intervals too narrow.',
               'reference': 'https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html'}, f, indent=2)
print('\nPAIRED MODEL DIFFERENCES — FIRST MINUS SECOND (percentage points)')
for row in pair_rows:
    print(f"{row['first_model']} - {row['second_model']} | {row['metric']} | "
          f"difference={row['difference_percentage_points']:+.2f} | "
          f"95% interval=[{row['ci95_low_pp']:+.2f}, {row['ci95_high_pp']:+.2f}] | "
          f"contains zero={bool(row['interval_contains_zero'])}")
print('\nIntervals describe test-sample uncertainty for these saved models, not variability across training seeds.')
print('Intervals containing zero do not establish that models are equivalent.')
print('The comparisons are exploratory; intervals have no multiple-comparison adjustment.')
print('Saved to:', followup_dir)
print('No models trained. Share the printed inspection and bootstrap output.')


All five recomputed metrics match saved scores for all three models.
Bootstrap: 1000/5000
Bootstrap: 2000/5000
Bootstrap: 3000/5000
Bootstrap: 4000/5000
Bootstrap: 5000/5000

PAIRED MODEL DIFFERENCES — FIRST MINUS SECOND (percentage points)
parsbert - xlm-r | typed_micro_f1 | difference=+1.00 | 95% interval=[-0.19, +2.17] | contains zero=True
parsbert - xlm-r | typed_macro_f1_six_classes | difference=+2.08 | 95% interval=[+0.25, +3.90] | contains zero=False
parsbert - xlm-r | boundary_micro_f1 | difference=+0.62 | 95% interval=[-0.34, +1.57] | contains zero=True
parsbert - xlm-r | PER_typed_f1 | difference=+1.18 | 95% interval=[+0.01, +2.40] | contains zero=False
parsbert - xlm-r | PER_boundary_recall | difference=+2.34 | 95% interval=[+1.28, +3.46] | contains zero=False
tookabert - xlm-r | typed_micro_f1 | difference=+1.35 | 95% interval=[+0.25, +2.43] | contains zero=False
tookabert - xlm-r | typed_macro_f1_six_classes | difference=+1.54 | 95% interval=[-0.30, +3.35] | contains zero=